# Instacart next-basket recommendations

Individual coursework by Edwige Mpore Rwagasana (confirmed by the author).

This portfolio copy preserves the existing analysis and saved results. Setup paths and presentation metadata were edited for sharing on November 5, 2026. Results are historical outputs from the source notebook; the analysis was not rerun for this packaging step. Course instructions remain to make the scope and provided scaffolding visible. See README.md and ../../PROVENANCE.md.


# Instacart project

## 1. Connect to Drive and load the data


In [ ]:
# Data files are supplied separately. See README.md.
from pathlib import Path
from IPython.display import display


In [ ]:
#Load dataset
import pandas as pd
import numpy as np


folder = str(Path("data"))

aisles = pd.read_csv(f"{folder}/aisles.csv")
departments = pd.read_csv(f"{folder}/departments.csv")
order_products = pd.read_csv(f"{folder}/order_products.csv")
orders = pd.read_csv(f"{folder}/orders.csv")
products = pd.read_csv(f"{folder}/products.csv")

# Double Check that everything loaded
datasets = {
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
    "orders": orders,
    "products": products
}

for name, data in datasets.items():
    print(f"{name}: {data.shape[0]:,} rows, {data.shape[1]} columns")
    display(data.head())

## 2. Understand the words and check the data
| Term from class | What it means here |
|---|---|
| User / customer | A person placing orders |
| Item / product | One grocery product |
| Basket | All products in one order |
| Fitting history / training history | Earlier purchases used to build recommendations |
| Validation basket | A later basket used to compare and tune models |
| Final test basket | The final `train` basket, saved for the end of the project |
| Candidate products | Products seen anywhere in fitting history that can be recommended |
| Cold-start product | A target product absent from all fitting history |
| Top-K | The first K products in an ordered recommendation list |
| Hit | A recommended product that appears in the actual next basket |
| Replenishment | Predicting products the customer bought before |
| Discovery | Predicting known products that this customer has not bought before |

`groupby` groups rows, `size` counts rows, `nunique` counts different IDs, `merge` joins tables by matching IDs, and `set` keeps unique values. `assert` stops the notebook if a check fails. A dictionary connects a key to its value, such as a customer ID to their recommendation list.

In [3]:
from IPython.display import display

items = order_products

# Each order ID should appear once in the orders table.
assert orders['order_id'].is_unique

# A product should not appear twice in the same basket.
assert not items.duplicated(['order_id', 'product_id']).any()

# Every purchased product and order must have a matching ID.
assert items['order_id'].isin(orders['order_id']).all()
assert items['product_id'].isin(products['product_id']).all()
print('Data checks passed. The tables can be joined safely.')

Data checks passed. The tables can be joined safely.


## 3. Validation split


In [4]:
def make_validation_split(orders, items, orders_back=1):
    prior_orders = orders[orders['eval_set'] == 'prior'].copy()

    # Find the validation order number separately for every customer.
    last_prior_number = prior_orders.groupby('user_id')['order_number'].transform('max')
    prior_orders['validation_number'] = last_prior_number - (orders_back - 1)

    # Fitting orders happen before the validation order.
    fitting_orders = prior_orders[
        prior_orders['order_number'] < prior_orders['validation_number']
    ].copy()

    # Keep customers with at least three fitting orders.
    order_counts = fitting_orders.groupby('user_id').size()
    customers_to_keep = order_counts[order_counts >= 3].index
    fitting_orders = fitting_orders[fitting_orders['user_id'].isin(customers_to_keep)]

    # Select exactly one validation order per remaining customer.
    is_target = prior_orders['order_number'] == prior_orders['validation_number']
    keep_customer = prior_orders['user_id'].isin(customers_to_keep)
    validation_orders = prior_orders[is_target & keep_customer].copy()

    # Attach customer IDs to product rows. Product rows do not already have user_id.
    order_columns = ['order_id', 'user_id', 'order_number']
    fitting_items = items.merge(fitting_orders[order_columns], on='order_id', validate='many_to_one')
    validation_items = items.merge(validation_orders[order_columns], on='order_id', validate='many_to_one')

    # Check that we always learn from earlier orders, never the target order.
    last_fitting_order = fitting_orders.groupby('user_id')['order_number'].max()
    target_order = validation_orders.set_index('user_id')['order_number']
    target_order = target_order.reindex(last_fitting_order.index)
    assert (last_fitting_order < target_order).all()
    assert validation_orders['user_id'].is_unique
    assert set(fitting_orders['order_id']).isdisjoint(set(validation_orders['order_id']))
    assert set(fitting_orders['user_id']) == set(validation_orders['user_id'])
    assert set(validation_items['order_id']) == set(validation_orders['order_id'])

    # A candidate is known somewhere in the fitting history, not necessarily to this customer.
    candidate_products = set(fitting_items['product_id'])
    validation_items['eligible'] = validation_items['product_id'].isin(candidate_products)

    # Store actual baskets as sets so a product is counted only once.
    full_baskets = validation_items.groupby('user_id')['product_id'].agg(set).to_dict()
    eligible_rows = validation_items[validation_items['eligible']]
    eligible_baskets = eligible_rows.groupby('user_id')['product_id'].agg(set).to_dict()

    # Keep related tables together; later functions read these named entries.
    return {
        'fitting_orders': fitting_orders,
        'validation_orders': validation_orders,
        'fitting_items': fitting_items,
        'validation_items': validation_items,
        'last_fitting_order': last_fitting_order,
        'candidate_products': candidate_products,
        'full_baskets': full_baskets,
        'eligible_baskets': eligible_baskets
    }

validation_data = make_validation_split(orders, items)
fitting_items = validation_data['fitting_items']
validation_items = validation_data['validation_items']
candidate_products = validation_data['candidate_products']

### 3a. Check the course numbers and choose K
A fitting basket contains about 9.89 products on average, so K=10 is a reasonable primary list length. This justification uses fitting purchases only.

In [5]:
actual_counts = {
    'Customers': len(validation_data['full_baskets']),
    'Fitting orders': len(validation_data['fitting_orders']),
    'Fitting product rows': len(fitting_items),
    'Candidate products': len(candidate_products),
    'Validation items': len(validation_items),
    'Cold-start items': int((~validation_items['eligible']).sum()),
    'Customers with eligible targets': len(validation_data['eligible_baskets'])
}
expected_counts = {
    'Customers': 5000, 'Fitting orders': 33230, 'Fitting product rows': 328487,
    'Candidate products': 24367, 'Validation items': 50638,
    'Cold-start items': 1270, 'Customers with eligible targets': 4994
}
assert actual_counts == expected_counts
check_table = pd.DataFrame({'Actual': actual_counts, 'Expected': expected_counts})
display(check_table)

# Count products inside each fitting order.
basket_sizes = fitting_items.groupby('order_id').size()
print('Mean fitting basket size:', round(basket_sizes.mean(), 2))
print('Median fitting basket size:', basket_sizes.median())
K = 10
print('Main recommendation list length:', K)

                                 Actual  Expected
Customers                          5000      5000
Fitting orders                    33230     33230
Fitting product rows             328487    328487
Candidate products                24367     24367
Validation items                  50638     50638
Cold-start items                   1270      1270
Customers with eligible targets    4994      4994

Mean fitting basket size: 9.89
Median fitting basket size: 8.0
Main recommendation list length: 10


The split matches every course checkpoint, so the comparisons use the intended purchase histories. A 10-product list is close to the average fitting basket of 9.89 products. However, 1,270 validation items were never seen in fitting history, so none of these models can recommend them under the current candidate rule.

## 4. Define and test the metrics



In [6]:
ranking_metrics = ['precision', 'recall', 'f1', 'ndcg']

def score_one_customer(recommendations, eligible_basket, full_basket_size, k=10):
    # Score only the first K recommendations.
    recommendations = list(recommendations)[:k]
    assert k > 0 and full_basket_size > 0
    assert len(recommendations) == len(set(recommendations))

    # No eligible products means ranking quality cannot be measured.
    if len(eligible_basket) == 0:
        return {'precision': np.nan, 'recall': np.nan, 'f1': np.nan,
                'ndcg': np.nan, 'end_recall': 0.0}

    # Record whether each recommended product was actually purchased.
    hits = []
    for product_id in recommendations:
        hits.append(product_id in eligible_basket)
    number_of_hits = sum(hits)

    # Empty slots in a short list still count as misses: divide by K.
    precision = number_of_hits / k
    recall = number_of_hits / len(eligible_basket)
    f1 = 0.0
    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)

    # Give more credit to hits near the top of the recommendation list.
    dcg = 0.0
    for position, is_hit in enumerate(hits, start=1):
        if is_hit:
            dcg += 1 / np.log2(position + 1)

    # The ideal list contains as many hits as possible, starting at position 1.
    ideal_dcg = 0.0
    possible_hits = min(k, len(eligible_basket))
    for position in range(1, possible_hits + 1):
        ideal_dcg += 1 / np.log2(position + 1)

    ndcg = dcg / ideal_dcg
    end_recall = number_of_hits / full_basket_size
    return {'precision': precision, 'recall': recall, 'f1': f1,
            'ndcg': ndcg, 'end_recall': end_recall}


def evaluate_model(recommendations_by_customer, data, k=10):
    # Build one results row for each customer in this validation split.
    customer_scores = {}
    for customer_id, full_basket in data['full_baskets'].items():
        recommendations = recommendations_by_customer.get(customer_id, [])
        eligible_basket = data['eligible_baskets'].get(customer_id, set())
        assert set(recommendations[:k]).issubset(data['candidate_products'])
        customer_scores[customer_id] = score_one_customer(
            recommendations, eligible_basket, len(full_basket), k
        )
    return pd.DataFrame.from_dict(customer_scores, orient='index').sort_index()

### 4a. Example


In [7]:
example_scores = score_one_customer([1, 2, 3], {1, 3, 4, 5}, 4, k=3)
display(pd.DataFrame([example_scores]).round(4))

# Show the calculation rather than treating NDCG as a black box.
positions = np.array([1, 2, 3])
weights = 1 / np.log2(positions + 1)
example_table = pd.DataFrame({
    'Position': positions, 'Recommended product': [1, 2, 3],
    'Hit': [True, False, True], 'Position weight': weights,
    'Actual credit': weights * np.array([1, 0, 1]),
    'Ideal credit': weights
})
display(example_table.round(4))
assert np.isclose(example_scores['precision'], 2 / 3)
assert np.isclose(example_scores['recall'], 2 / 4)
assert np.isclose(example_scores['f1'], 4 / 7)
assert np.isclose(example_scores['ndcg'], 1.5 / weights.sum())

# Also check short lists, small baskets, no hits and all-cold-start baskets.
assert score_one_customer([1], {1}, 1, 10)['precision'] == 0.1
assert score_one_customer([1], {1}, 1, 10)['ndcg'] == 1.0
assert score_one_customer([2], {1}, 1, 10)['ndcg'] == 0.0
assert np.isnan(score_one_customer([], set(), 1, 10)['ndcg'])
assert score_one_customer([], set(), 1, 10)['end_recall'] == 0.0
print('All metric checks passed.')

   precision  recall      f1    ndcg  end_recall
0     0.6667     0.5  0.5714  0.7039         0.5

   Position  Recommended product    Hit  Position weight  Actual credit  \
0         1                    1   True           1.0000            1.0   
1         2                    2  False           0.6309            0.0   
2         3                    3   True           0.5000            0.5   

   Ideal credit  
0        1.0000  
1        0.6309  
2        0.5000  

All metric checks passed.


 In this example, two of the three recommendations were useful, but they recovered only half of the four-product basket. F1=0.5714 balances these two outcomes; it does not mean 57.14% of products were predicted correctly. NDCG=0.7039 shows that the list also loses credit because one useful product appears below an incorrect recommendation.

## 5. Baselines
1. **Global popularity:** recommend products purchased by the most different fitting-history customers. Everyone gets the same list.
2. **Repeat frequency:** recommend products this customer bought most often.
3. **Recency-frequency:** also favor products the customer bought recently.

In [8]:
def build_purchase_history(data):
    # One row per customer-product pair.
    purchase_history = data['fitting_items'].groupby(['user_id', 'product_id']).agg(
        frequency=('order_id', 'size'),
        last_purchase_order=('order_number', 'max')
    ).reset_index()

    # 0 means bought in the latest fitting order; 1 means one order earlier.
    latest_order = purchase_history['user_id'].map(data['last_fitting_order'])
    purchase_history['orders_since_last'] = latest_order - purchase_history['last_purchase_order']
    return purchase_history


def make_ranked_lists(table, sort_columns, sort_directions, list_size=20):
    # First group customers together, then rank their products within each group.
    columns = ['user_id'] + sort_columns
    directions = [True] + sort_directions  # True: low to high; False: high to low.
    ranked_table = table.sort_values(columns, ascending=directions, kind='mergesort')

    # Keep the best products for each customer and save their ordered IDs.
    recommendations = {}
    for customer_id, customer_rows in ranked_table.groupby('user_id'):
        top_rows = customer_rows.head(list_size)
        recommendations[customer_id] = top_rows['product_id'].tolist()
    return recommendations


def recommend_repeat(purchase_history):
    # Highest purchase count first; smaller product ID breaks ties.
    return make_ranked_lists(purchase_history, ['frequency', 'product_id'], [False, True])


def recommend_recency(purchase_history, recency_weight):
    # Lambda=0 must give exactly the same order as repeat frequency.
    if recency_weight == 0:
        return recommend_repeat(purchase_history)

    scored_history = purchase_history.copy()
    decay = np.exp(-recency_weight * scored_history['orders_since_last'])
    scored_history['score'] = scored_history['frequency'] * decay

    # Break equal-score ties by count, last purchase, then product ID.
    return make_ranked_lists(
        scored_history,
        ['score', 'frequency', 'last_purchase_order', 'product_id'],
        [False, False, False, True]
    )


def build_baselines(data, recency_weight):
    purchase_history = build_purchase_history(data)

    # Count different customers, not total purchase rows, for global popularity.
    popularity = data['fitting_items'].groupby('product_id')['user_id'].nunique()
    popularity = popularity.reset_index(name='number_of_customers')
    popularity = popularity.sort_values(
        ['number_of_customers', 'product_id'], ascending=[False, True]
    )
    popular_products = popularity['product_id'].head(20).tolist()
    global_recommendations = {}
    for customer_id in data['full_baskets']:
        global_recommendations[customer_id] = popular_products.copy()

    return {
        'Global popularity': global_recommendations,
        'Repeat frequency': recommend_repeat(purchase_history),
        'Recency-frequency': recommend_recency(purchase_history, recency_weight)
    }

purchase_history = build_purchase_history(validation_data)
product_names = products.set_index('product_id')['product_name']
preview = purchase_history.head(10).copy()
preview['product_name'] = preview['product_id'].map(product_names)
display(preview)

   user_id  product_id  frequency  last_purchase_order  orders_since_last  \
0       43         196          2                    9                  1   
1       43        2180          1                    8                  2   
2       43        3452          2                   10                  0   
3       43        4605          3                    6                  4   
4       43        4920          2                   10                  0   
5       43        5223          1                    3                  7   
6       43        5322          1                    5                  5   
7       43        5952          1                    9                  1   
8       43        6352          1                    8                  2   
9       43        8204          2                    8                  2   

                                        product_name  
0                                               Soda  
1                                   Angel 

This table turns each customer's shopping history into two useful clues: how often they buy a product and how recently they bought it. For customer 43, yellow onions were bought three times but not in the last four fitting orders, while grapes were bought twice and in the latest order. Frequency alone favors onions; adding recency can give grapes higher priority.

### 5a. Tune lambda: how much should recent purchases matter?


In [9]:
lambda_values = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25,
                 0.30, 0.35, 0.40, 0.50, 0.75, 1.00]
tuning_rows = []
for recency_weight in lambda_values:
    recommendations = recommend_recency(purchase_history, recency_weight)
    customer_scores = evaluate_model(recommendations, validation_data, k=K)
    average_scores = customer_scores.mean().to_dict()
    average_scores['lambda'] = recency_weight
    tuning_rows.append(average_scores)

tuning_table = pd.DataFrame(tuning_rows)
columns = ['lambda', 'precision', 'recall', 'f1', 'ndcg', 'end_recall']
display(tuning_table[columns].round(4))

# idxmax returns the row with the highest validation NDCG.
best_row = tuning_table['ndcg'].idxmax()
best_lambda = float(tuning_table.loc[best_row, 'lambda'])
models = build_baselines(validation_data, best_lambda)
assert recommend_recency(purchase_history, 0) == models['Repeat frequency']
print('Selected lambda:', best_lambda)


    lambda  precision  recall      f1    ndcg  end_recall
0     0.00     0.2594  0.3359  0.2518  0.3889      0.3295
1     0.05     0.2685  0.3494  0.2615  0.4021      0.3427
2     0.10     0.2696  0.3503  0.2624  0.4037      0.3436
3     0.15     0.2700  0.3509  0.2628  0.4047      0.3443
4     0.20     0.2702  0.3512  0.2630  0.4050      0.3446
5     0.25     0.2719  0.3530  0.2645  0.4070      0.3464
6     0.30     0.2718  0.3530  0.2644  0.4070      0.3463
7     0.35     0.2713  0.3525  0.2639  0.4065      0.3459
8     0.40     0.2712  0.3524  0.2638  0.4064      0.3458
9     0.50     0.2704  0.3516  0.2631  0.4056      0.3451
10    0.75     0.2663  0.3468  0.2590  0.4005      0.3404
11    1.00     0.2656  0.3460  0.2584  0.3995      0.3398

Selected lambda: 0.25


Adding recency improves F1 from 0.2518 at lambda=0 to 0.2645 at lambda=0.25, and NDCG from 0.3889 to 0.4070. This suggests that recent purchases help make restocking suggestions more useful than purchase counts alone. Lambda=0.25 gives the best NDCG in this grid, although 0.30 is nearly tied.

### 5b. See how one customer's products receive scores


In [10]:
example_customer = int(purchase_history['user_id'].min())
customer_history = purchase_history[purchase_history['user_id'] == example_customer].copy()
customer_history['recency_multiplier'] = np.exp(-best_lambda * customer_history['orders_since_last'])
customer_history['score'] = customer_history['frequency'] * customer_history['recency_multiplier']
customer_history['product_name'] = customer_history['product_id'].map(product_names)
customer_history = customer_history.sort_values(
    ['score', 'frequency', 'last_purchase_order', 'product_id'],
    ascending=[False, False, False, True]
)
print('Customer:', example_customer)
display(customer_history[['product_name', 'frequency', 'orders_since_last',
                          'recency_multiplier', 'score']].head(15).round(3))

Customer: 43


                                         product_name  frequency  \
42                                             Banana          7   
79                                            Spinach          5   
73  Macaroni & Cheese Deluxe Four Cheese Sauce Dinner          6   
31                                      Fat Free Milk          4   
78                       Special K Red Berries Cereal          4   
19         100% Whole Wheat Pre-Sliced Sandwich Thins          3   
25           Naturally Hickory Smoked Thick Cut Bacon          3   
33                              Green Seedless Grapes          3   
50                                   Golden Pineapple          3   
65                 Cinnabon Cinnamon Rolls With Icing          3   
2        Southern Butter Pecan Gourmet Coffee Creamer          2   
4                                 Seedless Red Grapes          2   
20                                    Red Raspberries          2   
27                                       Strawbe

Customer 43's bananas receive the highest score because they were bought seven times, including the latest fitting order. Spinach ranks above macaroni and cheese even though macaroni was bought more often, because spinach was bought more recently. In practice, this rule would move frequently and recently purchased staples toward the top of a shopping reminder list.

## 6. Compare models: accuracy, coverage and concentration


In [11]:
def measure_coverage(recommendations_by_customer, candidates, k=10):
    # Count every recommendation occurrence across customers.
    recommended_products = []
    for recommendations in recommendations_by_customer.values():
        recommended_products.extend(recommendations[:k])
    counts = pd.Series(recommended_products, dtype='int64').value_counts()
    coverage = len(counts) / len(candidates)

    # Include candidate products that nobody received.
    counts = counts.reindex(sorted(candidates), fill_value=0)
    sorted_counts = np.sort(counts.to_numpy())
    total_recommendations = sorted_counts.sum()
    if total_recommendations == 0:
        return coverage, np.nan

    # Standard Gini formula: sorted positions weight the recommendation counts.
    number_of_products = len(sorted_counts)
    positions = np.arange(1, number_of_products + 1)
    weights = 2 * positions - number_of_products - 1
    gini = (weights * sorted_counts).sum() / (number_of_products * total_recommendations)
    return coverage, float(gini)


def compare_models(models, data, k=10):
    scores_by_model = {}
    average_rows = []
    for model_name, recommendations in models.items():
        customer_scores = evaluate_model(recommendations, data, k)
        scores_by_model[model_name] = customer_scores
        row = customer_scores.mean().to_dict()
        coverage, gini = measure_coverage(recommendations, data['candidate_products'], k)
        row['model'] = model_name
        row['coverage'] = coverage
        row['gini'] = gini
        average_rows.append(row)
    comparison_table = pd.DataFrame(average_rows).set_index('model')
    return scores_by_model, comparison_table

scores_by_model, comparison_table = compare_models(models, validation_data, k=K)
display(comparison_table.round(4))
leading_model = comparison_table['ndcg'].idxmax()
print('Highest validation NDCG@10:', leading_model)
print('Ranking customers:', scores_by_model[leading_model]['ndcg'].count())
print('End-to-end recall customers:', scores_by_model[leading_model]['end_recall'].count())

                   precision  recall      f1    ndcg  end_recall  coverage  \
model                                                                        
Global popularity     0.0668  0.0685  0.0602  0.0918      0.0676    0.0004   
Repeat frequency      0.2594  0.3359  0.2518  0.3889      0.3295    0.4495   
Recency-frequency     0.2719  0.3530  0.2645  0.4070      0.3464    0.4614   

                     gini  
model                      
Global popularity  0.9996  
Repeat frequency   0.8423  
Recency-frequency  0.8355  

Highest validation NDCG@10: Recency-frequency
Ranking customers: 4994
End-to-end recall customers: 5000


Recency-frequency gives the best balance of useful suggestions and basket recovery: F1=0.2645, compared with 0.2518 for repeat frequency and 0.0602 for global popularity. Its 27.2% precision means about 2.7 useful products per 10 recommendation slots on average, while its recall recovers 35.3% of the eligible basket. These results support personalized shopping reminders, but most of the next basket still goes unrecovered.

Recency-frequency also recommends 46.1% of the candidate catalog across customers, compared with only 10 products for global popularity (about 0.04%, rounded to 0.0% in the final printout). Its Gini of 0.8355 shows that recommendations still concentrate on some products. Broader coverage therefore means more varied lists, not an even spread across the whole catalog.

## 7. Check uncertainty

In [12]:
def paired_bootstrap(scores_a, scores_b, repeats=2000, seed=446):
    # Subtraction aligns customer IDs. Remove undefined paired scores.
    differences = (scores_a - scores_b).dropna().to_numpy()
    assert len(differences) > 0
    random_generator = np.random.default_rng(seed)
    bootstrap_means = []

    # Every repeat draws the same number of customers, with replacement.
    for repeat in range(repeats):
        sampled_differences = random_generator.choice(
            differences, size=len(differences), replace=True
        )
        bootstrap_means.append(sampled_differences.mean())

    lower, upper = np.percentile(bootstrap_means, [2.5, 97.5])
    return float(differences.mean()), float(lower), float(upper)


def interpret_difference(lower, upper, margin=0.01):
    if lower > margin:
        return 'meaningfully better'
    if upper < -margin:
        return 'meaningfully worse'
    if lower >= -margin and upper <= margin:
        return 'practically equivalent'
    return 'inconclusive'

comparison_rows = []
top_group = [leading_model]
for model_name in models:
    if model_name == leading_model:
        continue
    for metric in ranking_metrics:
        difference, lower, upper = paired_bootstrap(
            scores_by_model[leading_model][metric], scores_by_model[model_name][metric]
        )
        conclusion = 'No course practical margin for this metric'
        if metric == 'ndcg':
            conclusion = interpret_difference(lower, upper)
            if conclusion == 'practically equivalent':
                top_group.append(model_name)
        comparison_rows.append({
            'Comparison': leading_model + ' minus ' + model_name,
            'Metric': metric, 'Mean difference': difference,
            'Lower 95%': lower, 'Upper 95%': upper, 'Conclusion': conclusion
        })
bootstrap_table = pd.DataFrame(comparison_rows)
display(bootstrap_table.round(4))

# Choose the simplest baseline among the leader and equivalent models.
simplicity_order = ['Global popularity', 'Repeat frequency', 'Recency-frequency']
for model_name in simplicity_order:
    if model_name in top_group:
        chosen_model = model_name
        break
print('Top group:', top_group)
print('Provisional baseline choice:', chosen_model)

                                  Comparison     Metric  Mean difference  \
0  Recency-frequency minus Global popularity  precision           0.2050   
1  Recency-frequency minus Global popularity     recall           0.2845   
2  Recency-frequency minus Global popularity         f1           0.2043   
3  Recency-frequency minus Global popularity       ndcg           0.3152   
4   Recency-frequency minus Repeat frequency  precision           0.0125   
5   Recency-frequency minus Repeat frequency     recall           0.0172   
6   Recency-frequency minus Repeat frequency         f1           0.0127   
7   Recency-frequency minus Repeat frequency       ndcg           0.0181   

   Lower 95%  Upper 95%                                  Conclusion  
0     0.1995     0.2107  No course practical margin for this metric  
1     0.2763     0.2932  No course practical margin for this metric  
2     0.1992     0.2099  No course practical margin for this metric  
3     0.3074     0.3232            

Top group: ['Recency-frequency']
Provisional baseline choice: Recency-frequency


Recency-frequency's NDCG advantage over repeat frequency is 0.0181, with a 95% interval of 0.0157–0.0205. The entire interval is above the course's 0.01 practical margin, so the improvement is large enough to support choosing it over the simpler repeat rule. Precision, recall and F1 differences also favor recency-frequency, strengthening the validation evidence for this choice.

### 7a. Are nearby lambda values practically equivalent?
Compare each lambda with the best lambda using the same paired customer scores. A broad plateau means the value of recency matters more than the precise decimal.

In [13]:
lambda_comparison_rows = []
for recency_weight in lambda_values:
    recommendations = recommend_recency(purchase_history, recency_weight)
    customer_scores = evaluate_model(recommendations, validation_data, k=K)
    difference, lower, upper = paired_bootstrap(
        scores_by_model['Recency-frequency']['ndcg'], customer_scores['ndcg']
    )
    lambda_comparison_rows.append({
        'lambda': recency_weight, 'Best minus this lambda': difference,
        'Lower 95%': lower, 'Upper 95%': upper,
        'Conclusion': interpret_difference(lower, upper)
    })
display(pd.DataFrame(lambda_comparison_rows).round(4))

    lambda  Best minus this lambda  Lower 95%  Upper 95%  \
0     0.00                  0.0181     0.0157     0.0205   
1     0.05                  0.0049     0.0036     0.0063   
2     0.10                  0.0034     0.0022     0.0046   
3     0.15                  0.0023     0.0014     0.0033   
4     0.20                  0.0020     0.0012     0.0027   
5     0.25                  0.0000     0.0000     0.0000   
6     0.30                  0.0001    -0.0004     0.0006   
7     0.35                  0.0005    -0.0004     0.0013   
8     0.40                  0.0006    -0.0003     0.0015   
9     0.50                  0.0014     0.0002     0.0026   
10    0.75                  0.0065     0.0045     0.0085   
11    1.00                  0.0075     0.0054     0.0096   

                Conclusion  
0      meaningfully better  
1   practically equivalent  
2   practically equivalent  
3   practically equivalent  
4   practically equivalent  
5   practically equivalent  
6   practically 

Every nonzero lambda tested is practically equivalent to 0.25 under the course margin, while lambda=0 is meaningfully worse. The main lesson is to include recent shopping behavior; the precise weight matters less within this tested range. We can retain 0.25 as the selected value without claiming it is the only useful setting.

## 8. K sensitivity

In [14]:
list_lengths = [5, 10, 15, 20]
list_length_rows = []
list_length_comparisons = []
for list_length in list_lengths:
    scores_at_k, averages_at_k = compare_models(models, validation_data, k=list_length)
    for model_name in models:
        row = averages_at_k.loc[model_name].to_dict()
        row['model'] = model_name
        row['K'] = list_length
        list_length_rows.append(row)
        if model_name == leading_model:
            continue
        difference, lower, upper = paired_bootstrap(
            scores_at_k[leading_model]['ndcg'], scores_at_k[model_name]['ndcg']
        )
        list_length_comparisons.append({
            'K': list_length, 'Comparison': leading_model + ' minus ' + model_name,
            'Mean difference': difference, 'Lower 95%': lower, 'Upper 95%': upper,
            'Conclusion': interpret_difference(lower, upper)
        })
k_results = pd.DataFrame(list_length_rows)
for metric in ranking_metrics:
    print(metric.upper(), 'at each list length')
    display(k_results.pivot(index='K', columns='model', values=metric).round(4))
display(pd.DataFrame(list_length_comparisons).round(4))

PRECISION at each list length


model  Global popularity  Recency-frequency  Repeat frequency
K                                                            
5                 0.0865             0.3550            0.3373
10                0.0668             0.2719            0.2594
15                0.0546             0.2240            0.2123
20                0.0473             0.1911            0.1825

RECALL at each list length


model  Global popularity  Recency-frequency  Repeat frequency
K                                                            
5                 0.0456             0.2524            0.2400
10                0.0685             0.3530            0.3359
15                0.0821             0.4113            0.3922
20                0.0945             0.4512            0.4338

F1 at each list length


model  Global popularity  Recency-frequency  Repeat frequency
K                                                            
5                 0.0534             0.2504            0.2374
10                0.0602             0.2645            0.2518
15                0.0587             0.2539            0.2408
20                0.0569             0.2380            0.2276

NDCG at each list length


model  Global popularity  Recency-frequency  Repeat frequency
K                                                            
5                 0.1008             0.4231            0.4039
10                0.0918             0.4070            0.3889
15                0.0905             0.4122            0.3930
20                0.0932             0.4220            0.4039

    K                                 Comparison  Mean difference  Lower 95%  \
0   5  Recency-frequency minus Global popularity           0.3223     0.3135   
1   5   Recency-frequency minus Repeat frequency           0.0191     0.0161   
2  10  Recency-frequency minus Global popularity           0.3152     0.3074   
3  10   Recency-frequency minus Repeat frequency           0.0181     0.0157   
4  15  Recency-frequency minus Global popularity           0.3217     0.3138   
5  15   Recency-frequency minus Repeat frequency           0.0191     0.0170   
6  20  Recency-frequency minus Global popularity           0.3288     0.3209   
7  20   Recency-frequency minus Repeat frequency           0.0181     0.0159   

   Upper 95%           Conclusion  
0     0.3311  meaningfully better  
1     0.0222  meaningfully better  
2     0.3232  meaningfully better  
3     0.0205  meaningfully better  
4     0.3295  meaningfully better  
5     0.0213  meaningfully better  
6     0.3366  meaningfully 

For recency-frequency, increasing the list from 5 to 20 products raises recall from 25.2% to 45.1%, but lowers precision from 35.5% to 19.1%. A longer list finds more of the basket, while giving the customer more suggestions that they do not buy. F1 is highest at K=10 (0.2645), making 10 a reasonable balance here. Recency-frequency leads all four metrics at every tested list length, so its advantage does not depend on choosing exactly 10 products.

## 9. Diagnose behavior: repeat purchases versus discovery


In [15]:
diagnostic_items = validation_items.copy()

# Remember customer-product pairs seen in fitting history.
seen_pairs = set(zip(fitting_items['user_id'], fitting_items['product_id']))
item_types = []
previously_bought = []
for row in diagnostic_items.itertuples():
    bought_before = (row.user_id, row.product_id) in seen_pairs
    previously_bought.append(bought_before)
    if not row.eligible:
        item_types.append('cold start')
    elif bought_before:
        item_types.append('replenishment')
    else:
        item_types.append('discovery')
diagnostic_items['purchase_type'] = item_types

# Check reordered only AFTER making recommendations; it is not a model input.
assert np.array_equal(previously_bought, diagnostic_items['reordered'] == 1)


def item_hit_rates(recommendations_by_customer, group_column, k=10):
    # A set lets us quickly check if a customer received a product.
    recommended_pairs = set()
    for customer_id, recommendations in recommendations_by_customer.items():
        for product_id in recommendations[:k]:
            recommended_pairs.add((customer_id, product_id))

    hits = []
    for row in diagnostic_items.itertuples():
        hits.append((row.user_id, row.product_id) in recommended_pairs)
    scored_items = diagnostic_items.copy()
    scored_items['hit'] = hits
    return scored_items.groupby(group_column, observed=True)['hit'].mean()

purchase_type_table = pd.DataFrame()
for model_name, recommendations in models.items():
    purchase_type_table[model_name] = item_hit_rates(recommendations, 'purchase_type')
purchase_type_table['Target items'] = diagnostic_items.groupby('purchase_type').size()
display(purchase_type_table.round(4))

               Global popularity  Repeat frequency  Recency-frequency  \
purchase_type                                                           
cold start                0.0000            0.0000             0.0000   
discovery                 0.0374            0.0000             0.0000   
replenishment             0.0931            0.4843             0.5076   

               Target items  
purchase_type                
cold start             1270  
discovery             22621  
replenishment         26747  

Recency-frequency finds 50.8% of repeat-purchase item occurrences, compared with 48.4% for repeat frequency and 9.3% for global popularity. This makes it useful for reminding customers about products they may need again. However, both personal baselines find none of the 22,621 eligible discovery items because they recommend only products the customer bought before. Their strength is restocking; helping customers discover different products requires another approach.

In [16]:
fitting_order_counts = validation_data['fitting_orders'].groupby('user_id').size()
history_groups = pd.cut(
    fitting_order_counts, bins=[3, 5, 7, 10, 14], right=False,
    labels=['3-4 orders', '5-6 orders', '7-9 orders', '10-13 orders']
)
for model_name, customer_scores in scores_by_model.items():
    scores_with_groups = customer_scores.join(history_groups.rename('History length'))
    group_averages = scores_with_groups.groupby('History length', observed=True)[ranking_metrics].mean()
    print(model_name)
    display(group_averages.round(4))

Global popularity


                precision  recall      f1    ndcg
History length                                   
3-4 orders         0.0623  0.0697  0.0579  0.0890
5-6 orders         0.0662  0.0652  0.0597  0.0897
7-9 orders         0.0694  0.0695  0.0621  0.0942
10-13 orders       0.0714  0.0693  0.0618  0.0955

Repeat frequency


                precision  recall      f1    ndcg
History length                                   
3-4 orders         0.2366  0.3322  0.2359  0.3698
5-6 orders         0.2498  0.3353  0.2447  0.3815
7-9 orders         0.2765  0.3317  0.2631  0.3991
10-13 orders       0.2849  0.3469  0.2708  0.4141

Recency-frequency


                precision  recall      f1    ndcg
History length                                   
3-4 orders         0.2429  0.3424  0.2425  0.3807
5-6 orders         0.2651  0.3558  0.2604  0.4019
7-9 orders         0.2924  0.3504  0.2792  0.4198
10-13 orders       0.2995  0.3690  0.2855  0.4379

Recency-frequency's F1 rises from 0.2425 for customers with 3–4 fitting orders to 0.2855 for those with 10–13 orders; NDCG rises from 0.3807 to 0.4379. Customers with more history tend to receive better suggestions in these groups. The model still leads with shorter histories, but those customers remain harder to predict. This pattern does not prove that adding more orders alone causes the improvement.

 Popular products versus niche products


In [17]:
popularity = fitting_items.groupby('product_id')['user_id'].nunique()
popularity = popularity.reset_index(name='number_of_customers')
popularity = popularity.sort_values(
    ['number_of_customers', 'product_id'], ascending=[False, True]
).reset_index(drop=True)
popularity['popularity_rank'] = np.arange(1, len(popularity) + 1)
product_ranks = popularity.set_index('product_id')['popularity_rank']

diagnostic_items['Popularity group'] = pd.cut(
    diagnostic_items['product_id'].map(product_ranks),
    bins=[0, 100, 1000, 5000, np.inf],
    labels=['Top 100', '101-1,000', '1,001-5,000', 'Beyond 5,000']
)
diagnostic_items['Popularity group'] = diagnostic_items['Popularity group'].cat.add_categories('cold start')
diagnostic_items['Popularity group'] = diagnostic_items['Popularity group'].fillna('cold start')

popularity_table = pd.DataFrame()
for model_name, recommendations in models.items():
    popularity_table[model_name] = item_hit_rates(recommendations, 'Popularity group')
popularity_table['Target items'] = diagnostic_items.groupby('Popularity group', observed=True).size()
display(popularity_table.round(4))

# Product names make the global baseline easier to interpret.
popularity_preview = popularity.head(10).copy()
popularity_preview['product_name'] = popularity_preview['product_id'].map(product_names)
display(popularity_preview[['product_name', 'number_of_customers', 'popularity_rank']])

                  Global popularity  Repeat frequency  Recency-frequency  \
Popularity group                                                           
Top 100                      0.3131            0.3333             0.3531   
101-1,000                    0.0000            0.2625             0.2731   
1,001-5,000                  0.0000            0.2511             0.2609   
Beyond 5,000                 0.0000            0.2000             0.2120   
cold start                   0.0000            0.0000             0.0000   

                  Target items  
Popularity group                
Top 100                  10658  
101-1,000                14794  
1,001-5,000              14384  
Beyond 5,000              9532  
cold start                1270  

             product_name  number_of_customers  popularity_rank
0                  Banana                 1506                1
1  Bag of Organic Bananas                 1157                2
2    Organic Baby Spinach                  974                3
3    Organic Strawberries                  955                4
4         Organic Avocado                  827                5
5             Large Lemon                  809                6
6            Strawberries                  785                7
7                   Limes                  776                8
8    Organic Hass Avocado                  730                9
9     Organic Blueberries                  591               10

Recency-frequency finds 35.3% of target items in the top-100 popularity group and 21.2% among products ranked beyond 5,000. It can therefore suggest less common products when they matter to a particular customer, although its hit rate is lower for those products. Global popularity finds no items outside the top-100 group because its fixed top-10 list stays within that group. A personalized list can reflect individual shopping habits beyond widely purchased staples such as bananas.

In [18]:
example_customers = [int(fitting_order_counts.idxmin()), int(fitting_order_counts.idxmax())]
for customer_id in example_customers:
    print('Customer:', customer_id, '| Fitting orders:', fitting_order_counts.loc[customer_id])
    actual_basket = validation_data['full_baskets'][customer_id]
    for model_name, recommendations_by_customer in models.items():
        recommended_ids = recommendations_by_customer[customer_id][:K]
        rows = []
        for position, product_id in enumerate(recommended_ids, start=1):
            rows.append({'Rank': position, 'Recommended product': product_names.loc[product_id],
                         'Bought': product_id in actual_basket})
        print(model_name)
        display(pd.DataFrame(rows))
    print('Actual validation basket:', product_names.reindex(sorted(actual_basket)).tolist())

Customer: 252 | Fitting orders: 3
Global popularity


   Rank     Recommended product  Bought
0     1                  Banana   False
1     2  Bag of Organic Bananas   False
2     3    Organic Baby Spinach   False
3     4    Organic Strawberries   False
4     5         Organic Avocado   False
5     6             Large Lemon   False
6     7            Strawberries   False
7     8                   Limes   False
8     9    Organic Hass Avocado   False
9    10     Organic Blueberries   False

Repeat frequency


   Rank                                Recommended product  Bought
0     1                    Deli Fresh Smoked Turkey Breast   False
1     2                   Deli Sliced Sharp Cheddar Cheese    True
2     3                            Roasted Pine Nut Hummus   False
3     4                                     Romaine Hearts   False
4     5          Unsweetened Vanilla Coconut Milk Beverage   False
5     6  Sweeper Open Window Fresh Scent Wet Mopping Cl...   False
6     7                                  100% Oatnut Bread   False
7     8  Cleansing Towelettes Night Calming Makeup Remover   False
8     9                               Corn Maize Tortillas   False
9    10                    No Salt Added Whole Kernel Corn   False

Recency-frequency


   Rank                                Recommended product  Bought
0     1                    Deli Fresh Smoked Turkey Breast   False
1     2                            Roasted Pine Nut Hummus   False
2     3                                     Romaine Hearts   False
3     4                   Deli Sliced Sharp Cheddar Cheese    True
4     5          Unsweetened Vanilla Coconut Milk Beverage   False
5     6  Sweeper Open Window Fresh Scent Wet Mopping Cl...   False
6     7  Cleansing Towelettes Night Calming Makeup Remover   False
7     8                               Corn Maize Tortillas   False
8     9                    No Salt Added Whole Kernel Corn   False
9    10                   Organic Broccoli and Cauliflower   False

... (3 outputs left out)

   Rank          Recommended product  Bought
0     1                   Smartwater    True
1     2                       Banana   False
2     3           Orange Bell Pepper   False
3     4  Organic Vanilla Almond Milk   False
4     5               Blueberry Pint   False
5     6         Homemade Pizza Sauce   False
6     7         Organic Baby Spinach   False
7     8                        Limes   False
8     9           Whole Strawberries   False
9    10          Shredded Mozzarella   False

Recency-frequency


   Rank          Recommended product  Bought
0     1                   Smartwater    True
1     2                       Banana   False
2     3           Orange Bell Pepper   False
3     4             Organic Zucchini   False
4     5           Whole Strawberries   False
5     6                 Hass Avocado   False
6     7  Organic Vanilla Almond Milk   False
7     8         Homemade Pizza Sauce   False
8     9          Shredded Mozzarella   False
9    10                 Sweet Onions   False

Actual validation basket: ['Smartwater', 'Gluten Free Foods Granola', 'Shredded Mozzarella Cheese', 'Brussels Sprouts', 'Total 0% Nonfat Greek Yogurt', 'Large Lemon', 'Uncured Hickory Smoked Sunday Bacon']


For customer 252, global popularity finds no products, while both personal rules find the cheddar cheese. Repeat frequency places that hit second, while recency-frequency places it fourth, so recency does not improve every customer's list. For customer 2332, both personal rules put Smartwater first, while global popularity's only hit is a lemon in sixth place. These examples show why we check both the number of useful suggestions and where they appear, then judge the model across all customers.

## 10. Short-list sensitivity

In [19]:
popular_product_ids = popularity['product_id'].tolist()

def fill_short_lists(recommendations_by_customer, list_size=20):
    filled_lists = {}
    for customer_id in validation_data['full_baskets']:
        recommendations = list(recommendations_by_customer.get(customer_id, []))[:list_size]
        for product_id in popular_product_ids:
            if len(recommendations) >= list_size:
                break
            if product_id not in recommendations:
                recommendations.append(product_id)
        filled_lists[customer_id] = recommendations
    return filled_lists

filled_models = {}
for model_name in ['Repeat frequency', 'Recency-frequency']:
    filled_models[model_name + ' + popular fill'] = fill_short_lists(models[model_name])
filled_scores, filled_table = compare_models(filled_models, validation_data, k=K)
display(filled_table.round(4))
products_per_customer = purchase_history.groupby('user_id').size()
print('Customers with fewer than 10 past products:', int((products_per_customer < 10).sum()))


                                  precision  recall      f1    ndcg  \
model                                                                 
Repeat frequency + popular fill      0.2597  0.3369  0.2523  0.3893   
Recency-frequency + popular fill     0.2722  0.3541  0.2650  0.4075   

                                  end_recall  coverage    gini  
model                                                           
Repeat frequency + popular fill       0.3305    0.4495  0.8478  
Recency-frequency + popular fill      0.3474    0.4614  0.8412  

Customers with fewer than 10 past products: 440


Only 440 customers (8.8%) have fewer than 10 past products. Filling short lists increases recency-frequency's F1 only from 0.2645 to 0.2650 and NDCG from 0.4070 to 0.4075. Popular products can provide extra suggestions when history is limited, but the average gain is small and the leading model stays the same. The main conclusion does not depend on filling empty slots.

## 11. Earlier validation check

In [20]:
earlier_data = make_validation_split(orders, items, orders_back=2)
assert len(earlier_data['full_baskets']) == 4116
assert len(earlier_data['fitting_orders']) == 26462
assert len(earlier_data['candidate_products']) == 22493

# Rebuild models from the earlier history, without tuning lambda again.
earlier_models = build_baselines(earlier_data, best_lambda)
earlier_scores, earlier_table = compare_models(earlier_models, earlier_data, k=K)
print('Earlier validation results')
display(earlier_table.round(4))

# Restrict both periods to the same customers for a fairer comparison.
common_customers = sorted(set(validation_data['full_baskets']) & set(earlier_data['full_baskets']))
period_scores = {'Main validation': scores_by_model, 'Earlier validation': earlier_scores}
matched_rows = []
period_comparison_rows = []
for period_name, period_results in period_scores.items():
    for model_name, customer_scores in period_results.items():
        matched_scores = customer_scores.reindex(common_customers)
        row = matched_scores.mean().to_dict()
        row['Period'] = period_name
        row['Model'] = model_name
        matched_rows.append(row)
        if model_name == leading_model:
            continue
        leader_scores = period_results[leading_model].reindex(common_customers)
        difference, lower, upper = paired_bootstrap(leader_scores['ndcg'], matched_scores['ndcg'])
        period_comparison_rows.append({
            'Period': period_name, 'Comparison': leading_model + ' minus ' + model_name,
            'Mean difference': difference, 'Lower 95%': lower, 'Upper 95%': upper,
            'Conclusion': interpret_difference(lower, upper)
        })
display(pd.DataFrame(matched_rows).round(4))
display(pd.DataFrame(period_comparison_rows).round(4))

Earlier validation results


                   precision  recall      f1    ndcg  end_recall  coverage  \
model                                                                        
Global popularity     0.0664  0.0676  0.0598  0.0917      0.0665    0.0004   
Repeat frequency      0.2661  0.3381  0.2574  0.3962      0.3313    0.4404   
Recency-frequency     0.2802  0.3538  0.2708  0.4144      0.3465    0.4542   

                     gini  
model                      
Global popularity  0.9996  
Repeat frequency   0.8364  
Recency-frequency  0.8292  

   precision  recall      f1    ndcg  end_recall              Period  \
0     0.0681  0.0690  0.0611  0.0930      0.0681     Main validation   
1     0.2664  0.3386  0.2572  0.3956      0.3324     Main validation   
2     0.2803  0.3573  0.2713  0.4149      0.3510     Main validation   
3     0.0664  0.0676  0.0598  0.0917      0.0665  Earlier validation   
4     0.2661  0.3381  0.2574  0.3962      0.3313  Earlier validation   
5     0.2802  0.3538  0.2708  0.4144      0.3465  Earlier validation   

               Model  
0  Global popularity  
1   Repeat frequency  
2  Recency-frequency  
3  Global popularity  
4   Repeat frequency  
5  Recency-frequency  

               Period                                 Comparison  \
0     Main validation  Recency-frequency minus Global popularity   
1     Main validation   Recency-frequency minus Repeat frequency   
2  Earlier validation  Recency-frequency minus Global popularity   
3  Earlier validation   Recency-frequency minus Repeat frequency   

   Mean difference  Lower 95%  Upper 95%           Conclusion  
0           0.3219     0.3129     0.3314  meaningfully better  
1           0.0194     0.0169     0.0219  meaningfully better  
2           0.3227     0.3140     0.3317  meaningfully better  
3           0.0182     0.0157     0.0208  meaningfully better  

Recency-frequency still leads one order earlier, with F1=0.2708 versus 0.2574 for repeat frequency and NDCG=0.4144 versus 0.3962. On the same 4,116 customers, its NDCG is almost unchanged between the main and earlier checks (0.4149 versus 0.4144). Its paired advantage over repeat frequency remains above the practical margin in both periods, supporting a stable restocking baseline across these two validation points.

## Results

In [21]:
print('BASELINE DECISION')
print('Selected lambda:', best_lambda)
print('Primary validation leader:', leading_model)
print('Simplest baseline in the supported top group:', chosen_model)
print()
for model_name in models:
    row = comparison_table.loc[model_name]
    print(model_name)
    print('  Precision@10:', f"{row['precision']:.1%}")
    print('  Eligible-basket recall@10:', f"{row['recall']:.1%}")
    print('  NDCG@10:', f"{row['ndcg']:.4f}")
    print('  Candidate catalog coverage:', f"{row['coverage']:.1%}")

# Check which model leads each metric at each K.
print('\nMODEL RANKING CHECKS')
for list_length in list_lengths:
    rows_at_k = k_results[k_results['K'] == list_length].set_index('model')
    winners = rows_at_k[ranking_metrics].idxmax()
    print('K =', list_length, '| metric leaders:', winners.to_dict())
print('Earlier validation NDCG leader:', earlier_table['ndcg'].idxmax())

# Summarize why discovery remains a weakness of the personal baselines.
discovery_share = (diagnostic_items['purchase_type'] == 'discovery').mean()
cold_start_share = (diagnostic_items['purchase_type'] == 'cold start').mean()
print('\nBEHAVIOR INSIGHTS')
print('Eligible products new to the customer:', f'{discovery_share:.1%}', 'of target items')
print('Products absent from fitting history:', f'{cold_start_share:.1%}', 'of target items')
print('Compare discovery and replenishment hit rates in Section 9.')
print('Check paired intervals before treating a small score gap as meaningful.')

BASELINE DECISION
Selected lambda: 0.25
Primary validation leader: Recency-frequency
Simplest baseline in the supported top group: Recency-frequency

Global popularity
  Precision@10: 6.7%
  Eligible-basket recall@10: 6.9%
  NDCG@10: 0.0918
  Candidate catalog coverage: 0.0%
Repeat frequency
  Precision@10: 25.9%
  Eligible-basket recall@10: 33.6%
  NDCG@10: 0.3889
  Candidate catalog coverage: 44.9%
Recency-frequency
  Precision@10: 27.2%
  Eligible-basket recall@10: 35.3%
  NDCG@10: 0.4070
  Candidate catalog coverage: 46.1%

MODEL RANKING CHECKS
K = 5 | metric leaders: {'precision': 'Recency-frequency', 'recall': 'Recency-frequency', 'f1': 'Recency-frequency', 'ndcg': 'Recency-frequency'}
K = 10 | metric leaders: {'precision': 'Recency-frequency', 'recall': 'Recency-frequency', 'f1': 'Recency-frequency', 'ndcg': 'Recency-frequency'}
K = 15 | metric leaders: {'precision': 'Recency-frequency', 'recall': 'Recency-frequency', 'f1': 'Recency-frequency', 'ndcg': 'Recency-frequency'}
K = 2

**All in all:** Recency-frequency is the strongest baseline in these results. Its F1 of 0.2645 shows a better balance between useful recommendation slots and basket recovery than repeat frequency (0.2518) or global popularity (0.0602). It would be a reasonable starting point for personalized restocking reminders: the validation results show about 2.7 useful products per 10 slots and 34.6% average recovery of the full basket.

There is still substantial room to improve. Eligible products new to the customer account for 44.7% of target items, and the personal baselines miss all of them. Later models should be compared with this baseline and assessed for whether they can add useful discovery suggestions while retaining strong repeat-purchase recommendations. These findings describe validation performance; the final test baskets remain reserved for the end.

## 13. One addition: recommend related products
**Question:** Can we find useful products a customer has not bought before, while keeping good restocking suggestions?

We add one method, **item-item collaborative filtering**, inspired by Amazon and the class Jaccard example. Products are related when many of the same customers bought them during fitting history. This does not necessarily mean the products were in the same basket.

The same validation split, candidate products, K=10, and evaluation functions remain in use. The final test baskets stay sealed. Product names are used to explain results; they do not determine similarity.

There are four steps: find related products, make recommendations, compare results, and decide what the experiment tells us. This is a class-sized experiment, not a reproduction of Amazon's full system.

### 13a. Find related products
**Jaccard similarity** = customers who bought both products / customers who bought at least one of them.

Example: if 5 customers bought both yogurt and granola, and 20 bought at least one, similarity is 5/20=0.25.

For this first experiment, require at least **5 shared customers** and keep up to **25 related products per product**. These are fixed starting choices, not tuned optima. The 25 is the size of a product's related-product lookup; it is separate from the 10 suggestions shown to a customer.

The sparse matrix below stores only observed purchases. It lets us count shared customers without creating a huge table of every possible product pair.

In [22]:
from scipy.sparse import csr_matrix


def find_related_products(data, minimum_shared=5, number_related=25):
    # Count each customer-product pair once, even if it was purchased repeatedly.
    purchases = data['fitting_items'][['user_id', 'product_id']].drop_duplicates()
    purchasers_per_product = purchases.groupby('product_id').size()

    # A product with fewer than 5 purchasers cannot have 5 shared purchasers.
    supported_products = purchasers_per_product[purchasers_per_product >= minimum_shared].index
    purchases = purchases[purchases['product_id'].isin(supported_products)].copy()
    customer_ids = sorted(purchases['user_id'].unique())
    product_ids = np.array(sorted(purchases['product_id'].unique()))
    customer_positions = dict(zip(customer_ids, range(len(customer_ids))))
    product_positions = dict(zip(product_ids, range(len(product_ids))))

    # Matrix rows are customers; columns are products; 1 means purchased.
    rows = purchases['user_id'].map(customer_positions).to_numpy()
    columns = purchases['product_id'].map(product_positions).to_numpy()
    values = np.ones(len(purchases), dtype=np.int32)
    purchase_matrix = csr_matrix((values, (rows, columns)),
                                 shape=(len(customer_ids), len(product_ids)))

    # Matrix multiplication counts customers shared by each pair of products.
    shared_customers = (purchase_matrix.T @ purchase_matrix).tocoo()
    enough_support = shared_customers.data >= minimum_shared
    different_products = shared_customers.row != shared_customers.col
    keep = enough_support & different_products
    first = shared_customers.row[keep]
    second = shared_customers.col[keep]
    both = shared_customers.data[keep]

    # Convert shared counts into the Jaccard score taught in class.
    customer_counts = np.asarray(purchase_matrix.sum(axis=0)).ravel()
    at_least_one = customer_counts[first] + customer_counts[second] - both
    related_table = pd.DataFrame({
        'product_id': product_ids[first], 'related_product_id': product_ids[second],
        'shared_customers': both, 'similarity': both / at_least_one
    })
    related_table = related_table.sort_values(
        ['product_id', 'similarity', 'related_product_id'], ascending=[True, False, True]
    )
    related_table = related_table.groupby('product_id').head(number_related).copy()

    # Save a short related-product lookup for each product.
    related_lookup = {}
    for product_id, group in related_table.groupby('product_id'):
        related_lookup[product_id] = list(zip(group['related_product_id'], group['similarity']))
    return related_lookup, related_table

related_lookup, related_table = find_related_products(validation_data)

# Use names to inspect the related products for a commonly purchased item.
example_product = int(popularity.iloc[0]['product_id'])
example_related = related_table[related_table['product_id'] == example_product].head(10).copy()
example_related['Related product'] = example_related['related_product_id'].map(product_names)
print('Products related to:', product_names.loc[example_product])
display(example_related[['Related product', 'shared_customers', 'similarity']].round(4))

Products related to: Banana


             Related product  shared_customers  similarity
205526       Organic Avocado               386      0.1983
59723           Strawberries               375      0.1957
202759           Large Lemon               358      0.1829
77651   Organic Strawberries               378      0.1815
106560                 Limes               340      0.1751
83445   Organic Baby Spinach               346      0.1621
212599        Cucumber Kirby               273      0.1543
12350          Yellow Onions               259      0.1458
119254    Organic Fuji Apple               222      0.1285
14518    Seedless Red Grapes               219      0.1256

 Banana's related products are mostly common produce items. For example, 386 fitting-history customers bought both bananas and organic avocados. These shared purchasers provide a possible discovery clue, but they do not prove a customer needs both products in the next basket. The next steps test whether these relationships lead to useful suggestions.

### 13b. Make each customer's list
Use the customer's frequent and recent purchases as clues. Each past product gives points to its related products. Add those points, then rank the candidates.

This is an item-item method with frequency-and-recency weights on the customer's purchase history. It reuses the chosen lambda=0.25 rather than introducing another tuning search.

Previously purchased products remain eligible, so the list can contain both familiar and new-to-customer products. A product is not related to itself in the lookup, but it can receive points through other products in the customer's history. Lists with fewer than 20 scored products remain short, matching the main baseline policy.

In [23]:
def recommend_related_products(data, related_lookup, recency_weight):
    history = build_purchase_history(data)
    recommendations_by_customer = {}

    for customer_id, customer_history in history.groupby('user_id'):
        product_scores = {}
        for purchase in customer_history.itertuples():
            # Give frequent and recent purchases more influence.
            history_weight = purchase.frequency * np.exp(
                -recency_weight * purchase.orders_since_last
            )
            neighbors = related_lookup.get(purchase.product_id, [])
            for related_product, similarity in neighbors:
                points = history_weight * similarity
                product_scores[related_product] = product_scores.get(related_product, 0.0) + points

        # Highest total score first; product ID breaks ties consistently.
        ranked_products = sorted(product_scores, key=lambda p: (-product_scores[p], p))
        recommendations_by_customer[customer_id] = ranked_products[:20]

    return recommendations_by_customer

related_recommendations = recommend_related_products(validation_data, related_lookup, best_lambda)

# Show whether one customer's suggestions include products new to them.
customer_id = int(purchase_history['user_id'].min())
past_products = set(purchase_history[purchase_history['user_id'] == customer_id]['product_id'])
actual_basket = validation_data['full_baskets'][customer_id]
recommendation_rows = []
for rank, product_id in enumerate(related_recommendations[customer_id][:K], start=1):
    recommendation_rows.append({
        'Rank': rank, 'Product': product_names.loc[product_id],
        'Bought before': product_id in past_products,
        'In validation basket': product_id in actual_basket
    })
print('Customer:', customer_id)
display(pd.DataFrame(recommendation_rows))

Customer: 43


   Rank               Product  Bought before  In validation basket
0     1          Strawberries           True                 False
1     2                 Limes          False                 False
2     3        Cucumber Kirby          False                 False
3     4           Large Lemon          False                 False
4     5       Organic Avocado          False                 False
5     6  Organic Strawberries          False                 False
6     7  Organic Baby Spinach          False                 False
7     8         Yellow Onions           True                 False
8     9   Seedless Red Grapes           True                 False
9    10    Organic Fuji Apple          False                 False

Customer 43 receives both previously purchased and new-to-them products, so this method can generate discovery suggestions. None of the displayed top-10 suggestions appear in this customer's validation basket. Generating new options is only the first step; the options also need to be relevant.

### 13c. Compare with the baseline you already trust
Add one model to the same scorecard. Compare overall ranking quality and discovery separately. More discovery hits do not automatically mean a better complete shopping list.

The paired NDCG difference below is **item-item minus recency-frequency**. A negative difference means the new model scores lower. The same course practical margin applies at K=10.

In [24]:
# Keep the original models dictionary unchanged; create a comparison copy.
models_with_related = models.copy()
models_with_related['Item-item CF'] = related_recommendations
related_scores, related_summary = compare_models(models_with_related, validation_data, k=K)
display(related_summary.round(4))

# Reuse the existing item-type diagnostic; no new evaluation definition is needed.
related_behavior = pd.DataFrame()
for model_name in ['Recency-frequency', 'Item-item CF']:
    related_behavior[model_name] = item_hit_rates(models_with_related[model_name], 'purchase_type')
display(related_behavior.round(4))

cf_difference, cf_lower, cf_upper = paired_bootstrap(
    related_scores['Item-item CF']['ndcg'], scores_by_model['Recency-frequency']['ndcg']
)
print('Item-item minus recency-frequency NDCG:', round(cf_difference, 4))
print('95% interval:', round(cf_lower, 4), 'to', round(cf_upper, 4))
print('Course interpretation:', interpret_difference(cf_lower, cf_upper))

                   precision  recall      f1    ndcg  end_recall  coverage  \
model                                                                        
Global popularity     0.0668  0.0685  0.0602  0.0918      0.0676    0.0004   
Repeat frequency      0.2594  0.3359  0.2518  0.3889      0.3295    0.4495   
Recency-frequency     0.2719  0.3530  0.2645  0.4070      0.3464    0.4614   
Item-item CF          0.0589  0.0602  0.0526  0.0695      0.0592    0.0534   

                     gini  
model                      
Global popularity  0.9996  
Repeat frequency   0.8423  
Recency-frequency  0.8355  
Item-item CF       0.9946  

               Recency-frequency  Item-item CF
purchase_type                                 
cold start                0.0000        0.0000
discovery                 0.0000        0.0388
replenishment             0.5076        0.0772

Item-item minus recency-frequency NDCG: -0.3375
95% interval: -0.3455 to -0.3296
Course interpretation: meaningfully worse


This particular item-item model finds 3.9% of eligible discovery item occurrences, compared with 0% for the personal baseline. However, it finds only 7.7% of repeat-purchase occurrences, compared with 50.8% for recency-frequency. Its overall F1 is 0.0526 versus 0.2645, and NDCG@10 is 0.0695 versus 0.4070. The paired interval confirms a meaningful loss in overall ranking quality. Keep recency-frequency as the selected baseline; this experiment does not justify replacing it.

### 13d. Check list length and the earlier basket
Keep all item-item settings fixed. Reuse the four list lengths, then rebuild product relationships using the earlier fitting history only. This checks whether the new model's overall comparison depends on one list length or validation period.

In [25]:
related_k_rows = []
for list_length in list_lengths:
    for model_name in ['Recency-frequency', 'Item-item CF']:
        customer_scores = evaluate_model(models_with_related[model_name], validation_data, k=list_length)
        row = customer_scores.mean().to_dict()
        row['K'] = list_length
        row['model'] = model_name
        related_k_rows.append(row)
related_k_table = pd.DataFrame(related_k_rows)
display(related_k_table[['K', 'model', 'precision', 'recall', 'f1', 'ndcg']].round(4))

# Product relationships must be rebuilt from this fold's own fitting purchases.
earlier_related_lookup, earlier_related_table = find_related_products(earlier_data)
earlier_related_recommendations = recommend_related_products(
    earlier_data, earlier_related_lookup, best_lambda
)
earlier_related_scores = evaluate_model(earlier_related_recommendations, earlier_data, k=K)

# Compare the two models on the same customers across periods.
period_rows = []
for period_name, baseline_scores, cf_scores in [
    ('Main validation', scores_by_model['Recency-frequency'], related_scores['Item-item CF']),
    ('Earlier validation', earlier_scores['Recency-frequency'], earlier_related_scores)
]:
    for model_name, customer_scores in [('Recency-frequency', baseline_scores), ('Item-item CF', cf_scores)]:
        row = customer_scores.reindex(common_customers).mean().to_dict()
        row['Period'] = period_name
        row['Model'] = model_name
        period_rows.append(row)
display(pd.DataFrame(period_rows)[['Period', 'Model', 'precision', 'recall', 'f1', 'ndcg']].round(4))
print('No final-test baskets were used in this experiment.')

    K              model  precision  recall      f1    ndcg
0   5  Recency-frequency     0.3550  0.2524  0.2504  0.4231
1   5       Item-item CF     0.0629  0.0328  0.0382  0.0658
2  10  Recency-frequency     0.2719  0.3530  0.2645  0.4070
3  10       Item-item CF     0.0589  0.0602  0.0526  0.0695
4  15  Recency-frequency     0.2240  0.4113  0.2539  0.4122
5  15       Item-item CF     0.0559  0.0843  0.0599  0.0763
6  20  Recency-frequency     0.1911  0.4512  0.2380  0.4220
7  20       Item-item CF     0.0527  0.1064  0.0634  0.0839

               Period              Model  precision  recall      f1    ndcg
0     Main validation  Recency-frequency     0.2803  0.3573  0.2713  0.4149
1     Main validation       Item-item CF     0.0597  0.0598  0.0528  0.0694
2  Earlier validation  Recency-frequency     0.2802  0.3538  0.2708  0.4144
3  Earlier validation       Item-item CF     0.0598  0.0613  0.0534  0.0705

No final-test baskets were used in this experiment.


The item-item experiment stays below the recency-frequency baseline at every tested list length and in the earlier validation period. Its lower overall performance therefore is not limited to K=10 or the main basket. This finding applies to these fixed Jaccard, support and neighborhood choices; it does not show that every collaborative-filtering approach will perform poorly.